In [8]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output

JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# Updated to match YOUR Project One CRUD module file name and class name
from CRUD_Python_Module import CRUD
###########################


###########################
# Data Manipulation / Model
###########################
# Updated with your username/password from your test script screenshot

username = "aacuser"
password = "StrongPassword123!"

# If your CRUD constructor needs more args (host/port/db/collection), update here.
shelter = CRUD(username, password)


# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an
# invalid object type of 'ObjectID' - which will cause the data_table to crash - so we remove it
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

# Good safety step for Dash rendering
df.fillna('', inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash('SimpleExample')

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),

    # Add your unique identifier here so it shows in the screenshot
    html.Center(html.B(html.H1('SNHU CS-340 Dashboard - <Samari_Robinson_Camacho>'))),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),

        # REQUIRED: single row selection for map callback
        row_selectable="single",
        selected_rows=[0],

        # User-friendly features (good for rubric/client)
        page_size=10,
        sort_action="native",
        filter_action="native",

        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '120px', 'width': '120px', 'maxWidth': '180px'}
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        id='map-id',
        className='col s12 m6',
    )
])


#############################################
# Interaction Between Components / Controller
#############################################
# This callback will highlight a column on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    # Prevent errors on first load
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),

                # Marker with tool tip and popup
                # Column 13 and 14 define the grid-coordinates for the map
                # Column 4 defines the breed for the animal
                # Column 9 defines the name of the animal
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode
# If port 8050 is busy, use: app.run_server(port=8051)
app.run_server()


CRUD object created
Dash app running on https://bermudahusband-puzzledaniel-3000.codio.io/proxy/8050/
